# Notebook for Automatic Image Segmentation by Guiding SAM with Saliency Maps

This project proposes an automatic image segmentation by combining Saliency Maps with the Segment Anything Model (SAM). Utilizing a pretrained classification model, one can generate predictions and saliency maps using existing XAI methods, which are then employed to guide SAM segmentation.

## Environment Set-up

If running from Google Colab, set using_colab=True below and run the cell. In Colab, be sure to select 'GPU' under 'Edit'->'Notebook Settings'->'Hardware accelerator'.

In [ ]:
using_colab = True

In [ ]:
# Define your access token as secets and import it into notebook
from google.colab import userdata
access_token = userdata.get('access_token')

In [ ]:
if using_colab:
  import abc
  import json
  import os
  import random
  import sys

  from PIL import Image
  from matplotlib import pyplot as plt
  from skimage.transform import resize
  import numpy as np
  import pandas as pd
  import torch
  import torchvision
  from torchvision.datasets import VOCDetection
  from tqdm import tqdm

  # Delete the sample data directory and its contents
  !rm -r /content/sample_data

  print("PyTorch version:", torch.__version__)
  print("Torchvision version:", torchvision.__version__)
  print("CUDA is available:", torch.cuda.is_available())

  !{sys.executable} -m pip install opencv-python matplotlib
  !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'
  from segment_anything import sam_model_registry, SamPredictor

  # Download SAM Model file
  !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

  # Use git clone command to download the private repository
  !git clone {f'https://{access_token}@github.com/karray/Automatic-Image-Segmentation.git'}

  # Download imagenet_class_index file
  imagenet_file_path = 'imagenet_class_index.json'
  if not os.path.exists(imagenet_file_path):
      !wget https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json
  else: print("imagenet_class_index.json already exists")

  # Install and import captum
  !pip install captum > /dev/null 2>&1
  import captum
  print("Captum version:", captum.__version__)

  # Install and import torchcam
  !pip install torchcam > /dev/null 2>&1
  import torchcam
  print("Torchcam version:", torchcam.__version__)

  # Create the dataset directory
  !mkdir -p datasets/VOCSegmentation
  # Download the dataset
  dataset = torchvision.datasets.VOCSegmentation(
      root="datasets/VOCSegmentation",
      year="2012",
      image_set="val",
      download=True,
  )

## Imports

In [ ]:
sys.path.append('/content/Automatic-Image-Segmentation')

from model_handler.resnet import resnet
from dataset_handler.pascal import PascalDataset
from heatmap_handler import get_test_method
from model_handler import device
from utils.utils import SquareCropAndResize, imagenet_transform, inverse_transform, sam_predict
from prompt_handler.max_value import MaxValue
from utils.xai_visualizer import Visualizer
from segment_eval_handler.contained_segmentation import ContainedSegmentation

## Pipeline

Change root="/content/datasets/VOCSegmentation" in pascal.py

In [ ]:
# Handler Loader
model = resnet()
dataset = PascalDataset(device, imagenet_transform, inverse_transform, target_transform=SquareCropAndResize(224, interpolation=Image.NEAREST))
xai_methods = get_test_method()
point_generator = MaxValue()
segment_eval = ContainedSegmentation()

In [ ]:
# SAM
sam_checkpoint = '/content/sam_vit_b_01ec64.pth'
model_type = "vit_b"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam = sam.to(device)

In [ ]:
SAMPLES = 4
MULTIPLE_RESULTS = False

data = dataset.get_samples(SAMPLES)
for i in data:
    predictor = SamPredictor(sam)
    predictor.set_image(i.original_image)
    i.sam_predictor = predictor

In [ ]:
for indx, entry in enumerate(data):
    # init xai method
    method = xai_methods[0](model)
    # generate initial heatmap (and model classification label)
    method(entry)
    # generate points prompts (updates heatmap if interpolated)
    point_generator(entry)
    sam_predict(entry, MULTIPLE_RESULTS)
    segment_eval(entry)

In [ ]:
v = Visualizer(data)